# 🧠 Нейро-памятка по законам GTA 5 RP (San-Andreas)

Дообучение языковой модели на законах серверов **Memphis** и **Orlando**.
Модель отвечает кратко, понимает русский и английский, знает уровни розыска (★).

**Перед запуском:** `Среда выполнения → Сменить среду выполнения → GPU` (лучше A100/L4, подойдёт и T4).

Репозиторий загружается с GitHub пользователя **AloxinBay**.


## 1. Проверка GPU


In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'Включи GPU: Среда выполнения -> Сменить среду выполнения -> GPU'
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
cap = torch.cuda.get_device_capability(0)[0]
print(f'GPU: {gpu_name} | {gpu_mem:.0f} GB | compute capability {cap}.x')


## 2. Клонирование репозитория с GitHub (AloxinBay)


In [ ]:
# ⚠️ Укажи название своего репозитория на GitHub
GITHUB_USER = 'AloxinBay'
REPO_NAME   = 'neuro-laws'   # <-- замени на имя своего репозитория
REPO_URL = f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git'

import os, shutil
if os.path.exists(REPO_NAME):
    shutil.rmtree(REPO_NAME)
!git clone $REPO_URL
%cd $REPO_NAME
!ls


## 3. Установка зависимостей (закреплённые версии)


In [ ]:
!pip -q install \
  'transformers==4.44.2' \
  'trl==0.9.6' \
  'peft==0.12.0' \
  'accelerate==0.33.0' \
  'bitsandbytes==0.44.1' \
  'datasets==2.21.0' \
  'deep-translator==1.11.4' \
  'argostranslate==1.9.6' \
  'regex'
print('Зависимости установлены')


## 4. Парсинг законов и сборка датасета
`--translate` добавляет англоязычные пары через онлайн-переводчик (нужен интернет, есть в Colab).


In [ ]:
!python scripts/parse_laws.py --laws-dir laws --out data/laws.json
!python scripts/build_dataset.py --in data/laws.json --out data/train.jsonl --translate


## 5. Конфигурация обучения (авто-подстройка под GPU)


In [ ]:
# Базовая модель: мультиязычная, отлично знает русский и английский.
# 3B хватает для T4. На A100/L4 можно поставить 'Qwen/Qwen2.5-7B-Instruct'.
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

# Авто-настройка под доступную видеопамять (используем ресурсы по максимуму)
if gpu_mem >= 38:        # A100 40GB+
    BATCH, ACCUM, SEQ = 16, 1, 1024
elif gpu_mem >= 22:      # L4 / 3090 / A10
    BATCH, ACCUM, SEQ = 8, 2, 1024
else:                    # T4 16GB
    BATCH, ACCUM, SEQ = 4, 4, 1024

USE_BF16 = cap >= 8      # Ampere и новее
EPOCHS = 3
OUTPUT_DIR = 'outputs/lora'
print(f'model={MODEL_NAME} batch={BATCH} accum={ACCUM} seq={SEQ} bf16={USE_BF16} epochs={EPOCHS}')


## 6. Загрузка модели в 4-bit (QLoRA)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)


## 7. Подготовка датасета (применяем chat-шаблон модели)


In [ ]:
from datasets import load_dataset

raw = load_dataset('json', data_files='data/train.jsonl', split='train')
print('Примеров:', len(raw))

def to_text(example):
    return {'text': tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False)}

dataset = raw.map(to_text, remove_columns=raw.column_names)
print(dataset[0]['text'][:600])


## 8. Обучение (SFTTrainer, полная загрузка GPU)


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=25,
    save_strategy='epoch',
    bf16=USE_BF16,
    fp16=not USE_BF16,
    report_to='none',
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=SEQ,
    tokenizer=tokenizer,
    peft_config=lora_config,
    packing=True,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('LoRA-адаптер сохранён в', OUTPUT_DIR)


## 9. Слияние LoRA с базовой моделью (для удобного использования)


In [ ]:
import torch, gc
del model, trainer; gc.collect(); torch.cuda.empty_cache()

from peft import PeftModel
from transformers import AutoModelForCausalLM

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto')
merged = PeftModel.from_pretrained(base, OUTPUT_DIR)
merged = merged.merge_and_unload()

MERGED_DIR = 'outputs/merged'
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print('Готовая модель в', MERGED_DIR)


## 10. Проверка — задаём вопросы


In [ ]:
from transformers import pipeline
chat = pipeline('text-generation', model=merged, tokenizer=tokenizer,
                device_map='auto', max_new_tokens=256, do_sample=False)

SYSTEM = ('Тебя зовут MoonNeuro. Ты — юридический ассистент по законам GTA 5 RP штата San-Andreas. '
          'Отвечай кратко и по делу. ★ — это уровень розыска. '
          'Указывай номер статьи, части, суть нарушения и наказание. '
          'Понимай сленг игроков: госсник — полицейский, крайм — бандит, тулиться — стрелять. '
          'Основной язык — русский, но отвечай на языке вопроса.')

def ask(q):
    msgs = [{'role':'system','content':SYSTEM},{'role':'user','content':q}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    out = chat(prompt)[0]['generated_text'][len(prompt):]
    print(f'❓ {q}\n{out.strip()}\n' + '-'*50)

ask('12.8')
ask('наказание за 12.8 ч2')
ask('какая статья за убийство')
ask('what is article 12.8')


## 11. (Опционально) Сохранение на Google Drive или Hugging Face


In [ ]:
# --- Google Drive ---
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r outputs/merged /content/drive/MyDrive/neuro-laws-model

# --- Hugging Face Hub ---
# from huggingface_hub import login; login()
# merged.push_to_hub('AloxinBay/neuro-laws'); tokenizer.push_to_hub('AloxinBay/neuro-laws')
